![image_1779892225628.png](./image_1779892225628.png "image_1779892225628.png")

![image_1779892241170.png](./image_1779892241170.png "image_1779892241170.png")

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, IntegerType, StringType
from pyspark.sql import functions as f

# Initialize Spark
spark = SparkSession.builder.appName("ProductSalesDF").getOrCreate()

# -------------------------
# Product DataFrame
# -------------------------
product_schema = StructType([
    StructField("product_id", IntegerType(), True),
    StructField("product_name", StringType(), True),
    StructField("unit_price", IntegerType(), True)
])

product_data = [
    (1, "S8", 1000),
    (2, "G4", 800),
    (3, "iPhone", 1400)
]

product_df = spark.createDataFrame(product_data, schema=product_schema)

# -------------------------
# Sales DataFrame
# -------------------------
sales_schema = StructType([
    StructField("seller_id", IntegerType(), True),
    StructField("product_id", IntegerType(), True),
    StructField("buyer_id", IntegerType(), True),
    StructField("sale_date", StringType(), True),
    StructField("quantity", IntegerType(), True)
])

sales_data = [
    (1, 1, 1, "2019-01-21", 2),
    (1, 2, 2, "2019-02-17", 1),
    (2, 1, 3, "2019-06-02", 1),
    (3, 3, 3, "2019-05-13", 2),
    (2, 3, 1, "2019-03-08", 1)
]

sales_df = spark.createDataFrame(sales_data, schema=sales_schema)

# Convert sale_date to DateType (recommended)
sales_df = sales_df.withColumn("sale_date", to_date("sale_date", "yyyy-MM-dd"))

# Show DataFrames
product_df.show()
sales_df.show()

In [0]:
result_df=(
    product_df.join(sales_df,"product_id")
)
s8_df=(
    result_df
    .filter(f.col("product_name")=="S8")
    .select("buyer_id")
    .distinct()
)
iphone_df=(
    result_df
    .filter(f.col("product_name")=="iPhone")
    .select("buyer_id")
    .distinct()
)
display(result_df)
display(s8_df)
display(iphone_df)
final_df=(
    s8_df.subtract(iphone_df)
)
display(final_df)

In [0]:
from pyspark.sql import functions as f

final_df = (
    product_df
    .join(sales_df, "product_id")
    .filter(f.col("product_name").isin("S8", "iPhone"))
    .select("buyer_id", "product_name")
    .distinct()
    .groupBy("buyer_id")
    .agg(
        f.collect_set("product_name").alias("products")
    )
    .filter(
        (f.array_contains("products", "S8")) &
        (~f.array_contains("products", "iPhone"))
    )
    .select("buyer_id")
)

display(final_df)